[Markdown Cell]

# Notebook 00 — Kaggle + gated Hugging Face streaming

This version needs **no local copy of the 21GB raw corpus**. It reads the
official four JSON files over authenticated HTTP and stores only a bounded,
compressed selection of unique documents plus a disk-backed hash index.
Tokenizer learning, validation construction, and training-cache export then
reuse that selection locally, avoiding repeated remote corpus passes.

Default: 200M training BPE tokens, exactly 5M validation tokens, and a 32,768
entry Byte-level BPE vocabulary. Training proportions are 35% Supreme Court,
35% High Court, 15% Acts, 15% Tribunals. Validation has equal token quotas.

The default **budgeted audit** stops each source when enough text is retained;
it audits and deduplicates the consumed corpus, not unseen data. Set
`audit_full_corpus=True` to inspect all four complete files without storing
them. This uses more network time, not a 21GB raw download on disk.

This is CPU preprocessing. Dual-T4 DDP/NCCL, model parameter assertions, and
exact training-step checkpoint resume belong to Notebooks 01–04.

[Markdown Cell]

## 1. Kaggle setup

1. Enable **Internet** in notebook settings; use no accelerator for this run.
2. Under **Add-ons → Secrets**, attach a read-access HF token with label `HF`
   (the official starter's label) or `HF_TOKEN`. Accept the dataset's access
   agreement on Hugging Face before running.
3. Run cells in order. Do not paste or print the token. Authentication is read
   at runtime and never saved in configuration, metadata, or checkpoints.

The secret API is available inside Kaggle only. An `HF_TOKEN` environment
variable is supported for local testing or other runtimes.
[Kaggle secret instructions](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).

In [1]:
# [Code Cell]
%pip install --quiet "tokenizers==0.22.1" "ijson==3.4.0" "tqdm==4.67.1" "huggingface-hub==0.36.0" "requests==2.32.5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 34.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.3/148.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 13.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pand

In [2]:
# [Code Cell]
from __future__ import annotations

import gc
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import platform
import re
import shutil
import sqlite3
import sys
import tempfile
import time
import zlib
import unicodedata
from collections import Counter
from contextlib import contextmanager
from functools import lru_cache
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterator, Mapping, Sequence

import requests
from huggingface_hub import HfApi, hf_hub_url
from urllib3.exceptions import HTTPError as Urllib3HTTPError

import ijson
import numpy as np
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers
from tqdm.auto import tqdm

# Avoid uncontrolled Rust thread pools on a shared Kaggle CPU runtime.
os.environ["TOKENIZERS_PARALLELISM"] = "false"
PREPROCESS_VERSION = "cs781-notebook00-hf-stream-v2"

print({name: importlib.metadata.version(name)
       for name in ("numpy", "tokenizers", "ijson", "tqdm", "huggingface-hub", "requests")})
print("Python:", platform.python_version(), "| JSON backend:", ijson.backend)

{'numpy': '2.0.2', 'tokenizers': '0.22.1', 'ijson': '3.4.0', 'tqdm': '4.67.1', 'huggingface-hub': '0.36.0', 'requests': '2.32.5'}
Python: 3.12.13 | JSON backend: yajl2_c


[Markdown Cell]

## 2. Explicit data and storage budgets

The output directory is `/kaggle/working/`. The binaries themselves use only
400MB + 10MB with the default budgets. Compressed retained text is capped at
3GiB; SQLite at 6GiB; this job's files at 8GiB. Keep a 2GiB free-space margin.
The notebook checks actual free space instead of assuming a fixed Kaggle HDD
quota. Unrelated files and previous downloads still consume that free space.

Retain at most the source token quota in **words** for training, plus a small
validation reserve. Ordinary Byte-level regex pieces do not merge across
whitespace-separated words, so this is a conservative capacity allowance;
the final export verifies actual BPE token counts. A storage/capacity shortfall
fails with an actionable message; it never silently changes source weights.

`source_mode="local"` is optional for attached official files or QA. The
default is `"hf"`. No `hf_hub_download`, snapshot download, or raw-file cache
is used in the HF path.

In [3]:
# [Code Cell]
SOURCES = ("supreme_court", "high_courts", "acts", "tribunals")
SOURCE_FILES = {
    "supreme_court": "train_supreme_court.json", "high_courts": "train_high_courts.json",
    "acts": "train_acts.json", "tribunals": "train_tribunals.json",
}
TRAIN_PERCENT = dict(zip(SOURCES, (35, 35, 15, 15)))
SPECIAL_TOKENS = ("<pad>", "<s>", "</s>", "<unk>", "<sep>", "<mask>")
TOKEN_DTYPE = np.dtype("<u2")


@dataclass(frozen=True)
class PreprocessingConfig:
    """HF access, exact token budgets, and bounded CPU/disk working sets."""
    source_mode: str = "hf"
    hf_repo: str = "Exploration-Lab/CS-781-Capstone"
    hf_revision: str = "main"
    hf_secret_names: tuple[str, ...] = ("HF", "HF_TOKEN")
    hf_files: dict[str, str] = field(default_factory=dict)
    input_root: Path = Path("/kaggle/input")
    source_paths: dict[str, str] = field(default_factory=dict)
    json_prefixes: dict[str, str] = field(default_factory=dict)
    output_dir: Path = Path("/kaggle/working")
    train_tokens: int = 200_000_000
    val_tokens: int = 5_000_000
    vocab_size: int = 32_768
    tokenizer_sample_words: int = 4_000_000
    tokenizer_words_per_document: int = 8_192
    encode_chunk_words: int = 4_096
    prompt_words: int = 512
    reference_words: int = 128
    validation_reserve_factor: int = 2
    validation_candidate_percent: int = 20
    audit_full_corpus: bool = False
    max_cached_text_bytes: int = 3 << 30
    max_database_bytes: int = 6 << 30
    max_job_bytes: int = 8 << 30
    min_free_bytes: int = 2 << 30
    max_document_characters: int = 8_000_000
    commit_every_records: int = 500
    http_read_bytes: int = 65_536
    network_max_retries: int = 6
    connect_timeout: int = 20
    read_timeout: int = 120
    seed: int = 781

    def validate(self) -> None:
        """Reject incompatible quotas, unsafe ID storage, and invalid budgets."""
        assert self.source_mode in ("hf", "local")
        assert self.train_tokens > 0 and self.train_tokens % 100 == 0
        assert self.val_tokens > 0 and self.val_tokens % 4 == 0
        assert self.vocab_size == 32_768 <= np.iinfo(TOKEN_DTYPE).max + 1
        assert (self.prompt_words, self.reference_words) == (512, 128)
        assert self.tokenizer_sample_words >= 100
        assert self.tokenizer_words_per_document > 0 and self.encode_chunk_words > 0
        assert self.validation_reserve_factor >= 1 and 0 < self.validation_candidate_percent < 100
        assert 0 < self.max_cached_text_bytes < self.max_database_bytes < self.max_job_bytes
        assert self.min_free_bytes >= 0 and self.max_document_characters > 0
        assert self.commit_every_records > 0 and self.http_read_bytes >= 1_024
        assert self.network_max_retries > 0 and self.connect_timeout > 0 and self.read_timeout > 0
        assert set(self.source_paths) | set(self.hf_files) | set(self.json_prefixes) <= set(SOURCES)


cfg = PreprocessingConfig()
cfg.validate()
print(asdict(cfg))  # Configuration contains secret labels only, never token values.

{'source_mode': 'hf', 'hf_repo': 'Exploration-Lab/CS-781-Capstone', 'hf_revision': 'main', 'hf_secret_names': ('HF', 'HF_TOKEN'), 'hf_files': {}, 'input_root': PosixPath('/kaggle/input'), 'source_paths': {}, 'json_prefixes': {}, 'output_dir': PosixPath('/kaggle/working'), 'train_tokens': 200000000, 'val_tokens': 5000000, 'vocab_size': 32768, 'tokenizer_sample_words': 4000000, 'tokenizer_words_per_document': 8192, 'encode_chunk_words': 4096, 'prompt_words': 512, 'reference_words': 128, 'validation_reserve_factor': 2, 'validation_candidate_percent': 20, 'audit_full_corpus': False, 'max_cached_text_bytes': 3221225472, 'max_database_bytes': 6442450944, 'max_job_bytes': 8589934592, 'min_free_bytes': 2147483648, 'max_document_characters': 8000000, 'commit_every_records': 500, 'http_read_bytes': 65536, 'network_max_retries': 6, 'connect_timeout': 20, 'read_timeout': 120, 'seed': 781}


[Markdown Cell]

## 3. Canonical documents and bounded word helpers

Preserve case, punctuation, and citations; normalize to NFC and collapse
whitespace. The global deduplication key is
\(h(d)=\mathrm{SHA256}(\mathrm{UTF8}(\mathrm{canonicalize}(d)))\).
Whole-document hashes are computed before any retained prefix is cropped.
Duplicate ownership follows `SOURCES`; a held-out document's duplicate can
never enter training from another source.

In [4]:
# [Code Cell]
WHITESPACE = re.compile(r"\s+")
WORD_PATTERN = re.compile(r"\S+")


def canonicalize(text: str) -> str:
    """Return case-preserving NFC text with one space between words."""
    return WHITESPACE.sub(" ", unicodedata.normalize("NFC", text)).strip()


def document_hash(text: str) -> str:
    """Hash canonical UTF-8 text, not a source ID or Python's salted hash."""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def word_count(text: str) -> int:
    """Count words in canonical text without allocating text.split()."""
    return text.count(" ") + 1 if text else 0


def word_excerpt(text: str, start: int, count: int) -> str:
    """Extract a bounded word range without constructing every document word."""
    matches = itertools.islice(WORD_PATTERN.finditer(text), start, start + count)
    return " ".join(match.group(0) for match in matches)


def word_windows(text: str, width: int) -> Iterator[list[str]]:
    """Yield bounded, consecutive word lists; the final list may be shorter."""
    words = (match.group(0) for match in WORD_PATTERN.finditer(text))
    while window := list(itertools.islice(words, width)):
        yield window

[Markdown Cell]

## 4. Runtime-only authentication and immutable input identities

Resolve the repository once to a commit SHA and fetch file metadata. On
restart, continue that saved revision, even if `main` has advanced. Metadata
includes source sizes and LFS SHA-256 hashes when available, never credentials.
Completed ingestion can be reused offline without accessing Kaggle secrets.
[HF repository metadata](https://huggingface.co/docs/huggingface_hub/package_reference/hf_api#huggingface_hub.HfApi.dataset_info).

In [5]:
# [Code Cell]
@dataclass(frozen=True)
class SourceSpec:
    """Public immutable source identity; access credentials are never fields."""
    source: str
    filename: str
    size: int
    path_in_repo: str = ""
    revision: str = ""
    expected_sha256: str = ""
    local_path: str = ""
    repo_id: str = ""

    @property
    def name(self) -> str:
        return self.filename


@lru_cache(maxsize=1)
def get_hf_token(secret_names: tuple[str, ...]) -> str:
    """Read a token once without logging, persisting, or prompting for it."""
    token = os.environ.get("HF_TOKEN", "").strip()
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in secret_names:
            try:
                token = client.get_secret(name).strip()
                if token:
                    return token
            except Exception:
                continue
    except ImportError:
        pass
    raise RuntimeError("No HF token is available. In Kaggle Add-ons → Secrets, attach "
                       "a read token labelled HF or HF_TOKEN and enable Internet. "
                       "Outside Kaggle, set HF_TOKEN in the environment.") from None


def metadata_retry(cfg: PreprocessingConfig, token: str, revision: str) -> Any:
    """Fetch only repository metadata; redact transport exception details."""
    for attempt in range(cfg.network_max_retries):
        try:
            return HfApi(token=token).dataset_info(
                cfg.hf_repo, revision=revision, files_metadata=True, timeout=cfg.read_timeout
            )
        except Exception as exc:
            status = getattr(getattr(exc, "response", None), "status_code", None)
            if status in (401, 403, 404):
                raise RuntimeError(f"HF metadata access failed (HTTP {status}). Check the repository, "
                                   "token permissions, and accepted gated-dataset access.") from None
            if attempt + 1 == cfg.network_max_retries:
                raise RuntimeError("Cannot reach HF metadata after bounded retries. "
                                   "Enable Kaggle Internet and retry later.") from None
            time.sleep(min(2 ** attempt, 30))

In [6]:
# [Code Cell]
def discover_sources(cfg: PreprocessingConfig) -> dict[str, SourceSpec]:
    """Resolve four exact files without downloading any remote body."""
    if cfg.source_mode == "local":
        result: dict[str, SourceSpec] = {}
        for source, filename in SOURCE_FILES.items():
            paths = ([Path(cfg.source_paths[source])] if source in cfg.source_paths
                     else sorted(cfg.input_root.rglob(filename)))
            if len(paths) != 1 or not paths[0].is_file():
                raise FileNotFoundError(f"Expected exactly one {filename}; set source_paths explicitly.")
            result[source] = SourceSpec(source, filename, paths[0].stat().st_size,
                                        expected_sha256=file_sha256(paths[0]), local_path=str(paths[0]))
        return result
    info = metadata_retry(cfg, get_hf_token(cfg.hf_secret_names), cfg.hf_revision)
    if not info.sha:
        raise RuntimeError("HF did not return an immutable repository commit.")
    result = {}
    for source, filename in SOURCE_FILES.items():
        requested = cfg.hf_files.get(source)
        matches = [item for item in info.siblings if
                   (item.rfilename == requested if requested else Path(item.rfilename).name == filename)]
        if len(matches) != 1:
            raise FileNotFoundError(f"HF has {len(matches)} matches for {filename}. "
                                    "Set cfg.hf_files[source] to the exact repository path.")
        item = matches[0]
        size = item.size or getattr(item.lfs, "size", None)
        if not size or size <= 0:
            raise RuntimeError(f"No valid byte size in HF metadata for {filename}.")
        expected = getattr(item.lfs, "sha256", "") or ""
        result[source] = SourceSpec(source, filename, int(size), item.rfilename,
                                   info.sha, expected, repo_id=cfg.hf_repo)
    print("Pinned HF revision:", info.sha)
    print("Raw source total (not stored locally):", round(sum(s.size for s in result.values()) / 2**30, 2), "GiB")
    return result

[Markdown Cell]

## 5. Integrity and hard disk limits

Generated filenames are owned by this pipeline. Space checks include its
SQLite sidecars and partial outputs, and also check the filesystem's actual
free space. Nothing deletes unrelated raw downloads or Hugging Face caches.
If old downloads already fill the disk, clear those explicitly before running.

In [7]:
# [Code Cell]
def file_sha256(path: Path, show_progress: bool = False) -> str:
    """Hash a file in bounded blocks, optionally displaying byte progress."""
    digest = hashlib.sha256()
    with path.open("rb") as stream, tqdm(
        total=path.stat().st_size, unit="B", unit_scale=True,
        desc=f"Hash {path.name}", disable=not show_progress,
    ) as progress:
        while block := stream.read(8 << 20):
            digest.update(block)
            progress.update(len(block))
    return digest.hexdigest()


def sync_file(path: Path) -> None:
    """Request durable storage before publishing a generated file."""
    with path.open("r+b") as stream:
        os.fsync(stream.fileno())


def atomic_json(path: Path, payload: Any) -> None:
    """Write a small JSON artifact through a same-directory temporary file."""
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, default=str)
        stream.write("\n")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)

In [8]:
# [Code Cell]
FINAL_ARTIFACTS = (
    "tokenizer.json", "train.bin", "val.bin", "audit.json", "train_documents.jsonl",
    "validation_documents.jsonl", "validation_pairs.jsonl",
    "validation_prompts.json", "validation_truth.json",
)


def job_disk_bytes(cfg: PreprocessingConfig) -> int:
    """Count only this job's published/partial files and private work directory."""
    work = cfg.output_dir / "_preprocess"
    total = sum(p.stat().st_size for p in work.rglob("*") if p.is_file()) if work.exists() else 0
    for name in FINAL_ARTIFACTS + ("metadata.json",):
        for suffix in ("", ".partial"):
            path = cfg.output_dir / (name + suffix)
            if path.is_file():
                total += path.stat().st_size
    return total


def ensure_disk_space(directory: Path, additional_bytes: int, cfg: PreprocessingConfig) -> None:
    """Enforce both the job cap and an actual free-space reserve before writes."""
    free = shutil.disk_usage(directory).free
    used = job_disk_bytes(cfg)
    if free < additional_bytes + cfg.min_free_bytes or used + additional_bytes > cfg.max_job_bytes:
        raise OSError(f"Storage budget exceeded: job={used / 2**30:.2f} GiB, "
                      f"free={free / 2**30:.2f} GiB, job cap={cfg.max_job_bytes / 2**30:.2f} GiB. "
                      "Reduce train_tokens for a fresh run or explicitly clear unrelated old downloads. "
                      "No source proportions or validation budget were changed.")


def open_database(path: Path, cfg: PreprocessingConfig | None = None) -> sqlite3.Connection:
    """Bound the SQLite cache and physical database size; journal locally."""
    connection = sqlite3.connect(path)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA journal_mode=DELETE")
    connection.execute("PRAGMA synchronous=FULL")
    connection.execute("PRAGMA cache_size=-32768")
    connection.execute("PRAGMA temp_store=FILE")
    if cfg is not None:
        page_size = connection.execute("PRAGMA page_size").fetchone()[0]
        connection.execute(f"PRAGMA max_page_count={cfg.max_database_bytes // page_size}")
    return connection

[Markdown Cell]

## 6. Reliable HTTP connections and range validation

Request immutable resolve URLs with `stream=True` and identity encoding.
After a disconnect, reopen at the byte offset already delivered to the JSON
parser. Validate HTTP 206 Content-Range offsets and total size before using
bytes. If a server ignores Range and returns 200, discard the already-read
prefix in bounded chunks instead of duplicating it in the parser.

401/403/404 and incompatible content fail clearly; 429, transient 5xx,
connection failures, and timeouts use bounded retries and backoff. New resolve
requests obtain fresh signed redirects. Requests removes cross-host auth on
redirects by default. No raw response is materialized as `.content` or `.text`.
[HF streaming and ranges](https://huggingface.co/docs/hub/datasets-streaming),
[Requests response streaming](https://requests.readthedocs.io/en/latest/user/advanced/#body-content-workflow).

In [9]:
# [Code Cell]
class PermanentStreamError(RuntimeError):
    """Non-retryable identity, permission, or response-format failure."""


class TransientStreamError(RuntimeError):
    """A retryable response failure with an optional bounded wait."""
    def __init__(self, wait_seconds: float = 0.0) -> None:
        self.wait_seconds = min(max(wait_seconds, 0), 30)


class HTTPConnection:
    """Manage one authenticated response without writing any source bytes."""
    def __init__(self, spec: SourceSpec, token: str, cfg: PreprocessingConfig) -> None:
        self.spec, self.cfg = spec, cfg
        self.position = 0
        self.digest = hashlib.sha256()
        self.session = requests.Session()
        self.headers = {"Authorization": f"Bearer {token}", "Accept-Encoding": "identity"}
        self.response: Any = None
        self.closed = False
        self.reconnections = 0
        self.url = hf_hub_url(spec.repo_id, spec.path_in_repo, repo_type="dataset", revision=spec.revision)

    def close_response(self) -> None:
        if self.response is not None:
            self.response.close()
            self.response = None

    def connect(self) -> None:
        """Validate byte identity before a response can feed the parser."""
        self.close_response()
        headers = dict(self.headers, Range=f"bytes={self.position}-")
        response = self.session.get(self.url, headers=headers, stream=True,
                                    timeout=(self.cfg.connect_timeout, self.cfg.read_timeout))
        self.response = response
        status = response.status_code
        if status in (401, 403, 404):
            raise PermanentStreamError(f"{self.spec.name}: HTTP {status}; check HF read access and gating.")
        if status == 429 or 500 <= status < 600:
            retry_after = response.headers.get("Retry-After", "0")
            raise TransientStreamError(float(retry_after) if retry_after.isdigit() else 0)
        if status not in (200, 206):
            raise PermanentStreamError(f"{self.spec.name}: unexpected HTTP {status} at byte {self.position}.")
        if response.headers.get("Content-Encoding", "identity") not in ("", "identity"):
            raise PermanentStreamError("Server ignored identity encoding; byte-range integrity cannot be guaranteed.")
        if "text/html" in response.headers.get("Content-Type", "").lower():
            raise PermanentStreamError("Received an HTML access page instead of the JSON source.")
        if status == 206:
            match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", response.headers.get("Content-Range", ""))
            if not match or int(match[1]) != self.position or int(match[3]) != self.spec.size:
                raise PermanentStreamError("Content-Range does not match the pinned source and requested offset.")
        elif self.position:
            remaining = self.position
            while remaining:
                block = response.raw.read(min(remaining, self.cfg.http_read_bytes))
                if not block:
                    raise TransientStreamError()
                remaining -= len(block)
        self.reconnections += 1

    def close(self) -> None:
        self.close_response()
        self.session.close()
        self.closed = True

In [10]:
# [Code Cell]
class ResumableHTTPReader(HTTPConnection):
    """Bounded binary reads with offset-correct retry and streaming SHA-256."""
    def read(self, size: int = -1) -> bytes:
        if size == 0:
            return b""
        if self.closed:
            raise ValueError("Read from a closed stream")
        if self.position == self.spec.size:
            return b""
        if size < 0:
            raise ValueError("Unbounded HTTP reads are disabled; request a positive chunk size.")
        amount = min(size, self.cfg.http_read_bytes, self.spec.size - self.position)
        for attempt in range(self.cfg.network_max_retries):
            wait = min(2 ** attempt, 30)
            try:
                if self.response is None:
                    self.connect()
                block = self.response.raw.read(amount)
                if not block:
                    raise TransientStreamError()
                if len(block) > self.spec.size - self.position:
                    raise PermanentStreamError("Response exceeded the declared source byte length.")
                self.position += len(block)
                self.digest.update(block)
                return block
            except PermanentStreamError:
                self.close_response()
                raise
            except (requests.RequestException, Urllib3HTTPError, OSError, TransientStreamError) as exc:
                wait = max(wait, getattr(exc, "wait_seconds", 0))
                self.close_response()
                if attempt + 1 == self.cfg.network_max_retries:
                    raise RuntimeError(f"{self.spec.name}: network retries exhausted at byte {self.position}. "
                                       "Rerun to continue saved ingestion progress.") from None
                time.sleep(min(wait, 30))
        raise AssertionError("Unreachable retry state")


class LocalHashReader:
    """The same bounded hashing interface for optional attached-file inputs."""
    def __init__(self, spec: SourceSpec, cfg: PreprocessingConfig) -> None:
        self.stream = Path(spec.local_path).open("rb")
        self.position, self.digest, self.cfg = 0, hashlib.sha256(), cfg
        if Path(spec.local_path).stat().st_size != spec.size:
            self.stream.close()
            raise RuntimeError("Local source size changed since run initialization.")

    def read(self, size: int = -1) -> bytes:
        if size < 0:
            raise ValueError("Unbounded source reads are disabled")
        block = self.stream.read(min(size, self.cfg.http_read_bytes))
        self.position += len(block)
        self.digest.update(block)
        return block

    def close(self) -> None:
        self.stream.close()


@contextmanager
def open_source(spec: SourceSpec, cfg: PreprocessingConfig) -> Iterator[Any]:
    """Open at most one remote corpus response during ingestion; always close it."""
    reader = (LocalHashReader(spec, cfg) if spec.local_path else
              ResumableHTTPReader(spec, get_hf_token(cfg.hf_secret_names), cfg))
    try:
        yield reader
    finally:
        reader.close()

[Markdown Cell]

## 7. Stream JSON without seeking or manual brace parsing

The parser supports a root record array, an ID-to-record map, and root text
records/JSONL. Explicit `json_prefixes[source]` handles nested arrays such as
`"data.item"`. Both record strings and `text` lists of strings are accepted.
Malformed records fail; empty text is counted. Duplicate record fields are
rejected. The largest single JSON record still occupies RAM; a character guard
prevents continuing with unexpectedly huge documents. It cannot remove the
allocation inherent in parsing one scalar string.

Peek parser **events**, not bytes followed by `seek(0)`, so this works on a
live HTTP stream. UTF-8 and JSON syntax remain the parser's responsibility.
[ijson event interception](https://github.com/ICRAR/ijson).

In [11]:
# [Code Cell]
class StrictRecordDict(dict):
    """Reject duplicate fields inside a streamed JSON record."""

    def __setitem__(self, key: str, value: Any) -> None:
        if key in self:
            raise ValueError(f"Duplicate JSON record field: {key!r}")
        super().__setitem__(key, value)



class BOMReader:
    """Remove an optional UTF-8 BOM without seeking or losing source hashing."""
    def __init__(self, stream: Any) -> None:
        self.stream = stream
        self.prefix: bytes | None = None

    def read(self, size: int) -> bytes:
        if size == 0:
            return b""
        if self.prefix is None:
            prefix = b""
            while len(prefix) < 3:
                block = self.stream.read(3 - len(prefix))
                if not block:
                    break
                prefix += block
            self.prefix = prefix[3:] if prefix.startswith(b"\xef\xbb\xbf") else prefix
        take, self.prefix = self.prefix[:size], self.prefix[size:]
        return take + (self.stream.read(size - len(take)) if len(take) < size else b"")


def parse_json_stream(stream: Any, prefix: str | None = None) -> Iterator[tuple[int, str, Any]]:
    """Parse an unseekable binary stream: root array, ID map, or JSONL records."""
    events = iter(ijson.parse(BOMReader(stream), multiple_values=True, buf_size=65_536))
    first = next(events, None)
    if first is None:
        raise ValueError("Empty JSON source")
    second = next(events, None) if first[1] == "start_map" else None
    restored = itertools.chain([first], [second] if second is not None else [], events)
    if prefix is not None:
        records = ijson.items(restored, prefix, map_type=StrictRecordDict)
        yield from ((i, str(i), record) for i, record in enumerate(records))
    elif first[1] == "start_array":
        records = ijson.items(restored, "item", map_type=StrictRecordDict)
        yield from ((i, str(i), record) for i, record in enumerate(records))
    elif first[1] == "start_map" and second and second[1] == "map_key" and second[2] in ("text", "id", "document_id"):
        records = ijson.items(restored, "", map_type=StrictRecordDict)
        yield from ((i, str(i), record) for i, record in enumerate(records))
    elif first[1] == "start_map":
        records = ijson.kvitems(restored, "", map_type=StrictRecordDict)
        yield from ((i, str(key), record) for i, (key, record) in enumerate(records))
    else:
        raise ValueError("Expected a JSON array, ID map, or root text record")


def iter_json_records(path: Path) -> Iterator[tuple[int, str, Any]]:
    """Read small exported evaluator maps and local sanity fixtures."""
    with path.open("rb") as stream:
        yield from parse_json_stream(stream)


def extract_text(record: Any, path: Path, record_id: str, max_characters: int | None = None) -> str:
    """Extract only the document body, with strict schema validation."""
    value = record.get("text") if isinstance(record, dict) else record
    if isinstance(value, list) and all(isinstance(item, str) for item in value):
        if max_characters is not None and sum(map(len, value)) + len(value) > max_characters:
            raise ValueError(f"{path.name}: document exceeds max_document_characters before normalization")
        value = " ".join(value)
    if not isinstance(value, str):
        raise TypeError(f"{path.name}, ID={record_id!r}: text must be a string "
                        "or a list of strings; inspect the input schema")
    if max_characters is not None and len(value) > max_characters:
        raise ValueError(f"{path.name}: document exceeds max_document_characters before normalization")
    return canonicalize(value)

[Markdown Cell]

## 8. Small parser sanity checks

In [12]:
# [Code Cell]
assert canonicalize("  Section\t12\n applies. ") == "Section 12 applies."
assert document_hash(canonicalize("Cafe\u0301\nAct")) == document_hash("Café Act")
assert list(word_windows("one two three four five", 2)) == [
    ["one", "two"], ["three", "four"], ["five"]
]
assert word_excerpt("one two three four", 1, 2) == "two three"

cfg.output_dir.mkdir(parents=True, exist_ok=True)
with tempfile.TemporaryDirectory(dir=cfg.output_dir) as temporary:
    test_path = Path(temporary) / "fixture.json"
    for payload in ([{"text": "Section 12"}], {"case_1": {"text": ["Section", "12"]}}):
        test_path.write_text(json.dumps(payload), encoding="utf-8")
        rows = list(iter_json_records(test_path))
        assert len(rows) == 1
        assert extract_text(rows[0][2], test_path, rows[0][1]) == "Section 12"
print("Parser and normalization checks passed.")

Parser and normalization checks passed.


[Markdown Cell]

## 9. Persistent deduplication and compressed retained text

SQLite retains only hashes and compact audit metadata for unselected records.
Selected text is compressed per document with zlib. Each source has a
deterministic hash-selected validation candidate pool; those documents never
contribute to tokenizer learning. Training retains bounded prefixes, and
separate cached-text SHA-256 values verify decompression later.

Ingestion stats and document inserts commit together every 500 records.
If a session restarts, completed sources are reused; the unfinished JSON
source is replayed from its beginning, verifying a rolling document-hash chain
through the committed prefix. This avoids storing raw JSON or assuming that
an arbitrary byte offset is a valid JSON record boundary. In-session network
failures resume byte ranges without replaying parsed documents.

In [13]:
# [Code Cell]
def create_index(connection: sqlite3.Connection) -> None:
    """Create disk-backed audit/selection state without retaining the corpus."""
    connection.executescript("""
        CREATE TABLE IF NOT EXISTS documents (
            sha256 TEXT PRIMARY KEY, source TEXT NOT NULL, ordinal INTEGER NOT NULL,
            record_id TEXT NOT NULL, words INTEGER NOT NULL, priority TEXT NOT NULL,
            validation_candidate INTEGER NOT NULL DEFAULT 0, split TEXT NOT NULL DEFAULT 'train',
            sample_start INTEGER NOT NULL DEFAULT 0, sample_words INTEGER NOT NULL DEFAULT 0,
            cached_text BLOB, cached_sha256 TEXT, cached_words INTEGER NOT NULL DEFAULT 0,
            fully_cached INTEGER NOT NULL DEFAULT 0, cached_utf8_bytes INTEGER NOT NULL DEFAULT 0
        );
        CREATE TABLE IF NOT EXISTS source_progress (source TEXT PRIMARY KEY, data TEXT NOT NULL, complete INTEGER NOT NULL);
        CREATE TABLE IF NOT EXISTS validation_documents (sha256 TEXT PRIMARY KEY);
        CREATE TABLE IF NOT EXISTS training_documents (sha256 TEXT PRIMARY KEY);
        CREATE INDEX IF NOT EXISTS retained_source_ordinal ON documents(source, ordinal) WHERE cached_text IS NOT NULL;
        CREATE INDEX IF NOT EXISTS retained_source_priority ON documents(source, priority) WHERE cached_text IS NOT NULL;
    """)
    connection.commit()


def source_targets(source: str, cfg: PreprocessingConfig) -> tuple[int, int]:
    """Conservative training-word allowance and whole-window validation reserve."""
    train_words = cfg.train_tokens * TRAIN_PERCENT[source] // 100
    width = cfg.prompt_words + cfg.reference_words
    reserve_words = math.ceil((cfg.val_tokens // 4) * cfg.validation_reserve_factor / width) * width
    return train_words, reserve_words


def empty_source_stats() -> dict[str, Any]:
    """Durable source-local counters and rolling replay identity."""
    return dict(records=0, empty=0, duplicates=0, unique_documents=0, unique_words=0,
                eligible_validation_documents=0, retained_train_words=0, retained_validation_words=0,
                retained_compressed_bytes=0, chain="00" * 32, raw_bytes_read=0,
                raw_sha256="", full_file_read=False, parse_complete=False, complete=False)


def commit_ingestion(connection: sqlite3.Connection, source: str, stats: dict[str, Any], complete: bool) -> None:
    """Atomically commit document writes and the exact processed-record cursor."""
    stats["complete"] = complete
    connection.execute("INSERT OR REPLACE INTO source_progress VALUES (?, ?, ?)",
                       (source, json.dumps(stats), int(complete)))
    connection.commit()

In [14]:
# [Code Cell]
def retain_document(text: str, sha256: str, words: int, stats: dict[str, Any],
                    source: str, cfg: PreprocessingConfig) -> tuple[Any, str, int, int, int, int]:
    """Return compressed prefix, cache hash, words, candidate flag, completeness."""
    train_target, reserve_target = source_targets(source, cfg)
    width = cfg.prompt_words + cfg.reference_words
    bucket = int(hashlib.sha256(f"val:{cfg.seed}:{sha256}".encode()).hexdigest()[:16], 16) % 100
    candidate = (words >= width and bucket < cfg.validation_candidate_percent and
                 stats["retained_validation_words"] < reserve_target)
    if candidate:
        take = min((words // width) * width, reserve_target - stats["retained_validation_words"])
    elif stats["retained_train_words"] < train_target:
        take = min(words, train_target - stats["retained_train_words"])
    else:
        return None, "", 0, 0, 0, 0
    if take <= 0:
        return None, "", 0, 0, 0, 0
    cached = text if take == words else word_excerpt(text, 0, take)
    utf8 = cached.encode("utf-8")
    compressed = zlib.compress(utf8, level=3)
    assert word_count(cached) == take
    return compressed, document_hash(cached), take, int(candidate), int(take == words), len(utf8)


def record_chain(previous: bytes, record_id: str, sha256: str) -> bytes:
    """Bind a replay checkpoint to the ordered canonical document identities."""
    record_bytes = record_id.encode("utf-8")
    return hashlib.sha256(previous + len(record_bytes).to_bytes(8, "big") + record_bytes +
                          bytes.fromhex(sha256)).digest()


def source_budget_ready(stats: dict[str, Any], source: str, cfg: PreprocessingConfig) -> bool:
    train_target, reserve_target = source_targets(source, cfg)
    return stats["retained_train_words"] >= train_target and stats["retained_validation_words"] >= reserve_target

In [15]:
# [Code Cell]
def ingest_source(connection: sqlite3.Connection, spec: SourceSpec,
                  cfg: PreprocessingConfig) -> dict[str, Any]:
    """Stream once, resume committed prefixes, and never persist raw responses."""
    saved = connection.execute("SELECT data, complete FROM source_progress WHERE source=?", (spec.source,)).fetchone()
    if saved and saved["complete"]:
        return json.loads(saved["data"])
    stats = json.loads(saved["data"]) if saved else empty_source_stats()
    committed_records, expected_chain = stats["records"], stats["chain"]
    chain = bytes(32)
    other_bytes = sum(json.loads(row[0])["retained_compressed_bytes"] for row in
                      connection.execute("SELECT data FROM source_progress WHERE source!=?", (spec.source,)))
    parse_complete = False
    with open_source(spec, cfg) as reader, tqdm(desc=f"Stream {spec.source}", unit="doc") as progress:
        records = parse_json_stream(reader, cfg.json_prefixes.get(spec.source))
        try:
            for ordinal, record_id, record in records:
                text = extract_text(record, spec, record_id, max_characters=cfg.max_document_characters)
                del record
                if len(text) > cfg.max_document_characters:
                    raise ValueError(f"{spec.name}: one document exceeds max_document_characters. "
                                     "Inspect it and raise the guard only if RAM permits.")
                sha256 = document_hash(text)
                chain = record_chain(chain, record_id, sha256)
                progress.update(1)
                if ordinal < committed_records:
                    if ordinal + 1 == committed_records and chain.hex() != expected_chain:
                        raise RuntimeError("Replayed source prefix differs from saved ingestion. "
                                           "Use a fresh output directory; cached text is not trusted.")
                    continue
                assert ordinal == stats["records"]
                stats["records"] += 1
                stats["chain"] = chain.hex()
                if not text:
                    stats["empty"] += 1
                elif connection.execute("SELECT 1 FROM documents WHERE sha256=?", (sha256,)).fetchone():
                    stats["duplicates"] += 1
                else:
                    words = word_count(text)
                    compressed, cached_hash, cached_words, candidate, fully_cached, cached_utf8_bytes = retain_document(
                        text, sha256, words, stats, spec.source, cfg
                    )
                    size = len(compressed) if compressed is not None else 0
                    if other_bytes + stats["retained_compressed_bytes"] + size > cfg.max_cached_text_bytes:
                        raise OSError("Compressed text cache cap reached. Lower train_tokens for a fresh run; "
                                      "the 21GB raw corpus was not downloaded.")
                    if size:
                        ensure_disk_space(cfg.output_dir, size * 2 + 65_536, cfg)
                    priority = hashlib.sha256(f"{cfg.seed}:{sha256}".encode()).hexdigest()
                    connection.execute("INSERT INTO documents (sha256, source, ordinal, record_id, words, "
                                       "priority, validation_candidate, cached_text, cached_sha256, cached_words, fully_cached, cached_utf8_bytes) "
                                       "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
                                       (sha256, spec.source, ordinal, record_id, words, priority, candidate,
                                        compressed, cached_hash, cached_words, fully_cached, cached_utf8_bytes))
                    stats["unique_documents"] += 1
                    stats["unique_words"] += words
                    stats["eligible_validation_documents"] += int(words >= 640)
                    stats["retained_compressed_bytes"] += size
                    stats["retained_validation_words" if candidate else "retained_train_words"] += cached_words
                if stats["records"] % cfg.commit_every_records == 0:
                    commit_ingestion(connection, spec.source, stats, False)
                    ensure_disk_space(cfg.output_dir, 0, cfg)
                    progress.set_postfix(unique=stats["unique_documents"], dup=stats["duplicates"])
                if not cfg.audit_full_corpus and source_budget_ready(stats, spec.source, cfg):
                    break
            else:
                parse_complete = True
        finally:
            records.close()
        if stats["records"] < committed_records:
            raise RuntimeError("Source ended before its committed replay prefix.")
        stats.update(raw_bytes_read=reader.position, raw_sha256=reader.digest.hexdigest(),
                     full_file_read=reader.position == spec.size, parse_complete=parse_complete)
        if stats["full_file_read"] and spec.expected_sha256 and stats["raw_sha256"] != spec.expected_sha256:
            raise RuntimeError("Complete source SHA-256 mismatch. Use a fresh output directory "
                               "after resolving the source integrity problem.")
        if not stats["unique_documents"] or not stats["retained_train_words"] or not stats["retained_validation_words"]:
            raise ValueError(f"{spec.name}: insufficient unique training/validation text. "
                             "Check the corpus and candidate percentage; no source weights were changed.")
        commit_ingestion(connection, spec.source, stats, True)
    return stats

[Markdown Cell]

## 10. Select train-only tokenizer excerpts from the retained cache

Tokenizer input is capped at 4M words in the requested source proportions.
Hash priorities choose bounded excerpts within retained training text; this
does not claim a uniform sample of the unseen 21GB corpus. All validation
candidates are excluded. The trainer retains frequency/merge tables, so an
iterator alone does not make BPE training constant-memory.

In [16]:
# [Code Cell]
def plan_document_groups(connection: sqlite3.Connection, cfg: PreprocessingConfig) -> dict[str, Any]:
    """Plan a bounded BPE sample without re-reading any remote file."""
    connection.execute("UPDATE documents SET sample_start=0, sample_words=0 WHERE sample_words>0")
    plan: dict[str, Any] = {}
    for source in SOURCES:
        target_words = cfg.tokenizer_sample_words * TRAIN_PERCENT[source] // 100
        sampled_words = 0
        rows = connection.execute("SELECT sha256, cached_words FROM documents WHERE source=? "
                                  "AND cached_text IS NOT NULL AND validation_candidate=0 ORDER BY priority", (source,))
        try:
            for row in rows:
                count = min(row["cached_words"], cfg.tokenizer_words_per_document, target_words - sampled_words)
                start = int(row["sha256"][:16], 16) % (row["cached_words"] - count + 1)
                connection.execute("UPDATE documents SET sample_start=?, sample_words=? WHERE sha256=?",
                                   (start, count, row["sha256"]))
                sampled_words += count
                if sampled_words >= target_words:
                    break
        finally:
            rows.close()
        if not sampled_words:
            raise ValueError(f"{source}: no retained text available for tokenizer training")
        plan[source] = {"tokenizer_target_words": target_words, "tokenizer_actual_words": sampled_words}
    connection.commit()
    assert connection.execute("SELECT COUNT(*) FROM documents WHERE validation_candidate=1 AND sample_words>0").fetchone()[0] == 0
    ensure_disk_space(cfg.output_dir, 0, cfg)
    return plan


def iter_selected_documents(connection: sqlite3.Connection, path: Any, source: str,
                            predicate: str, cfg: PreprocessingConfig) -> Iterator[tuple[sqlite3.Row, str]]:
    """Yield and verify one compressed retained document at a time.

    path is kept for a uniform source-helper signature; no source is reopened.
    predicate is an internal SQL fragment, never user-supplied SQL.
    """
    cursor = connection.execute(f"SELECT * FROM documents WHERE source=? AND cached_text IS NOT NULL "
                                f"AND ({predicate}) ORDER BY ordinal", (source,))
    try:
        for row in cursor:
            expected_bytes = row["cached_utf8_bytes"]
            assert 0 < expected_bytes <= cfg.max_document_characters * 4
            inflater = zlib.decompressobj()
            utf8 = inflater.decompress(row["cached_text"], expected_bytes + 1)
            assert len(utf8) == expected_bytes and inflater.eof
            assert not inflater.unused_data and not inflater.unconsumed_tail
            text = utf8.decode("utf-8")
            assert document_hash(text) == row["cached_sha256"], "Retained text cache failed integrity verification"
            assert word_count(text) == row["cached_words"]
            if row["fully_cached"]:
                assert row["cached_sha256"] == row["sha256"]
            yield row, text
    finally:
        cursor.close()

[Markdown Cell]

## 11. Learn the dedicated Byte-level BPE

BPE merges frequent adjacent byte-symbol pairs inside regex pieces. Supply
the complete 256-byte alphabet and the six control tokens. There is no
pretrained vocabulary and no automatic special-token postprocessor.
`add_prefix_space=False` preserves round trips; explicit BOS/EOS insertion
is controlled by the export code.
[HF Byte-level BPE construction](https://pypi.org/project/tokenizers/0.22.1/).

In [17]:
# [Code Cell]
def train_legal_tokenizer(connection: sqlite3.Connection, paths: Mapping[str, Path],
                          tokenizer_path: Path, cfg: PreprocessingConfig) -> Tokenizer:
    """Learn a byte-complete BPE vocabulary from train-only streamed excerpts."""
    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=cfg.vocab_size, min_frequency=2, show_progress=True,
        special_tokens=list(SPECIAL_TOKENS),
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), max_token_length=64,
    )
    total_words = connection.execute("SELECT SUM(sample_words) FROM documents").fetchone()[0]

    def training_texts() -> Iterator[str]:
        with tqdm(total=total_words, desc="BPE sample", unit="word") as progress:
            for source in SOURCES:
                for row, text in iter_selected_documents(connection, paths[source], source, "sample_words>0", cfg):
                    assert not row["validation_candidate"]
                    excerpt = word_excerpt(text, row["sample_start"], row["sample_words"])
                    assert word_count(excerpt) == row["sample_words"]
                    yield excerpt
                    progress.update(row["sample_words"])

    tokenizer.train_from_iterator(training_texts(), trainer=trainer)
    if tokenizer.get_vocab_size() != cfg.vocab_size:
        raise ValueError("The corpus sample did not produce exactly 32,768 vocabulary entries. "
                         "Increase tokenizer_sample_words in a fresh output directory.")
    temporary = tokenizer_path.with_name(tokenizer_path.name + ".partial")
    tokenizer.save(str(temporary))
    sync_file(temporary)
    os.replace(temporary, tokenizer_path)
    return Tokenizer.from_file(str(tokenizer_path))

In [18]:
# [Code Cell]
def verify_tokenizer(tokenizer: Tokenizer, cfg: PreprocessingConfig) -> dict[str, int]:
    """Assert exact vocabulary size, stable control IDs, and Unicode round trips."""
    assert tokenizer.get_vocab_size() == cfg.vocab_size == 32_768
    assert len(set(tokenizer.get_vocab().values())) == cfg.vocab_size
    special_ids = {token: tokenizer.token_to_id(token) for token in SPECIAL_TOKENS}
    assert list(special_ids.values()) == list(range(len(SPECIAL_TOKENS)))
    for text in ("Section 12(1): ₹500; Café — न्यायालय.", "[2024] 3 SCC 123", "<mask> text <sep>"):
        encoded = tokenizer.encode(text, add_special_tokens=False).ids
        assert encoded and all(0 <= token < cfg.vocab_size for token in encoded)
        assert tokenizer.decode(encoded, skip_special_tokens=False) == text
    return special_ids

[Markdown Cell]

## 12. Exact-size, atomically published memory maps

Use raw little-endian uint16 files without a NumPy header. Check ID bounds
before conversion, flush, and publish only after the exact target is written.
Incomplete `.partial` files cannot be mistaken for completed caches.
[NumPy memory mapping](https://numpy.org/doc/stable/reference/generated/numpy.memmap.html).

In [19]:
# [Code Cell]
class AtomicTokenWriter:
    """Write bounded ID chunks into an exact-size, atomically published memmap."""

    def __init__(self, path: Path, count: int, vocab_size: int) -> None:
        self.path = path
        self.temporary = path.with_name(path.name + ".partial")
        self.expected = count
        self.vocab_size = vocab_size
        self.written = 0
        self.array: np.memmap | None = np.memmap(
            self.temporary, dtype=TOKEN_DTYPE, mode="w+", shape=(count,)
        )

    def __enter__(self) -> "AtomicTokenWriter":
        return self

    def write(self, token_ids: Sequence[int]) -> None:
        """Append one small chunk; reject invalid IDs and budget overflow."""
        ids = np.asarray(token_ids, dtype=np.int64)
        assert ids.ndim == 1 and ids.size > 0
        assert int(ids.min()) >= 0 and int(ids.max()) < self.vocab_size
        end = self.written + ids.size
        assert end <= self.expected and self.array is not None
        self.array[self.written:end] = ids.astype(TOKEN_DTYPE, copy=False)
        self.written = int(end)

    def close(self) -> None:
        """Drop the only mapping reference; no exported views are retained."""
        if self.array is not None:
            self.array.flush()
            array, self.array = self.array, None
            del array
            gc.collect()

    def __exit__(self, exc_type: Any, exc: Any, traceback: Any) -> bool:
        self.close()
        if exc_type is None:
            if self.written != self.expected:
                raise ValueError(f"Incomplete {self.path.name}: {self.written}/{self.expected} tokens")
            sync_file(self.temporary)
            os.replace(self.temporary, self.path)
        return False

[Markdown Cell]

## 13. Balanced validation and intact continuation examples

Each source supplies exactly 1,250,000 stored tokens. Nonoverlapping windows
contain 512 prompt words plus 128 reference words. Every contributing original
document is fully held out, including its uncached remainder and duplicates.

An exact BPE token total generally does not equal an integer number of complete
640-word windows. Only the final **binary-cache** slice for each source may be
cropped. The JSON examples remain intact; lengths, offsets, and truncation
flags make the distinction explicit. Loss evaluation uses `val.bin`, while
continuation scoring uses the complete prompt/reference maps.

In [20]:
# [Code Cell]
def build_validation(connection: sqlite3.Connection, paths: Mapping[str, Path],
                     tokenizer: Tokenizer, cfg: PreprocessingConfig) -> dict[str, Any]:
    """Write exact balanced token quotas and complete continuation JSONL rows."""
    quota = cfg.val_tokens // len(SOURCES)
    width = cfg.prompt_words + cfg.reference_words
    bos, eos = tokenizer.token_to_id("<s>"), tokenizer.token_to_id("</s>")
    pairs_path = cfg.output_dir / "validation_pairs.jsonl"
    docs_path = cfg.output_dir / "validation_documents.jsonl"
    pairs_tmp, docs_tmp = Path(str(pairs_path) + ".partial"), Path(str(docs_path) + ".partial")
    connection.execute("DELETE FROM validation_documents")
    connection.execute("UPDATE documents SET split='train'")
    connection.commit()
    stats: dict[str, Any] = {}
    with AtomicTokenWriter(cfg.output_dir / "val.bin", cfg.val_tokens, cfg.vocab_size) as writer, \
            pairs_tmp.open("w", encoding="utf-8") as pairs_file, \
            docs_tmp.open("w", encoding="utf-8") as docs_file:
        for source in SOURCES:
            counts = Counter(cached_tokens=0, full_example_tokens=0, examples=0, documents=0, truncated_examples=0)
            selected = iter_selected_documents(connection, paths[source], source, "validation_candidate=1", cfg)
            try:
                with tqdm(total=quota, desc=f"Validation {source}", unit="tok") as progress:
                    for row, text in selected:
                        contributed = False
                        for window_index, words in enumerate(word_windows(text, width)):
                            if len(words) != width:
                                continue
                            prompt = " ".join(words[:cfg.prompt_words])
                            reference = " ".join(words[cfg.prompt_words:])
                            ids = [bos] + tokenizer.encode(prompt + " " + reference, add_special_tokens=False).ids + [eos]
                            stored = min(len(ids), quota - counts["cached_tokens"])
                            if not contributed:
                                assert row["sample_words"] == 0
                                connection.execute("INSERT INTO validation_documents VALUES (?)", (row["sha256"],))
                                docs_file.write(json.dumps({"sha256": row["sha256"], "source": source,
                                                            "ordinal": row["ordinal"], "words": row["words"], "cached_words": row["cached_words"]}) + "\n")
                                counts["documents"] += 1
                                contributed = True
                            example = {
                                "id": f"{source}:{row['sha256']}:{window_index}", "source": source,
                                "document_sha256": row["sha256"], "word_offset": window_index * width,
                                "prompt": prompt, "reference": reference, "cache_offset": writer.written,
                                "cache_length": stored, "full_token_length": len(ids),
                                "cache_truncated": stored < len(ids),
                            }
                            pairs_file.write(json.dumps(example, ensure_ascii=False) + "\n")
                            writer.write(ids[:stored])
                            counts.update(cached_tokens=stored, full_example_tokens=len(ids), examples=1,
                                          truncated_examples=int(stored < len(ids)))
                            progress.update(stored)
                            if counts["cached_tokens"] == quota:
                                break
                        if counts["cached_tokens"] == quota:
                            break
            finally:
                selected.close()
            if counts["cached_tokens"] != quota:
                raise ValueError(f"{source}: validation pool produced {counts['cached_tokens']:,}/{quota:,} tokens. "
                                 "Attach all official data or increase validation_reserve_factor in a fresh run.")
            stats[source] = dict(counts)
    for temporary, final in ((pairs_tmp, pairs_path), (docs_tmp, docs_path)):
        sync_file(temporary)
        os.replace(temporary, final)
    connection.execute("UPDATE documents SET split='val' WHERE sha256 IN (SELECT sha256 FROM validation_documents)")
    connection.commit()
    return stats

[Markdown Cell]

## 14. Streaming exports for the official evaluator

The local Phase-1 evaluator expects matching ID-to-record JSON objects with a `text` field. Export those maps without assembling a large dictionary.

In [21]:
# [Code Cell]
def iter_jsonl(path: Path) -> Iterator[dict[str, Any]]:
    """Read one exported JSONL record at a time."""
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                yield json.loads(line)


def export_evaluation_map(pairs_path: Path, output_path: Path, field_name: str) -> None:
    """Write {example_id: {'text': selected_text}} without accumulating records."""
    temporary = output_path.with_name(output_path.name + ".partial")
    with temporary.open("w", encoding="utf-8") as stream:
        stream.write("{\n")
        first = True
        for example in iter_jsonl(pairs_path):
            if not first:
                stream.write(",\n")
            first = False
            stream.write(json.dumps(example["id"]) + ": " +
                         json.dumps({"text": example[field_name]}, ensure_ascii=False))
        stream.write("\n}\n")
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, output_path)

[Markdown Cell]

## 15. Bounded training tokenization

Encode at word boundaries and carry the actual leading space into subsequent
chunks. Add BOS once per original document. EOS is emitted only when retained
text reaches the true document end; an uncached remainder is not mislabeled
as an end of document. The final source-quota crop is recorded in the manifest.

In [22]:
# [Code Cell]
def document_token_chunks(text: str, tokenizer: Tokenizer,
                          cfg: PreprocessingConfig, emit_eos: bool = True) -> Iterator[tuple[list[int], bool]]:
    """Yield ([IDs], is_document_end), with one BOS/EOS per original document."""
    yield [tokenizer.token_to_id("<s>")], False
    first = True
    for words in word_windows(text, cfg.encode_chunk_words):
        chunk = ("" if first else " ") + " ".join(words)
        first = False
        yield tokenizer.encode(chunk, add_special_tokens=False).ids, False
    if emit_eos:
        yield [tokenizer.token_to_id("</s>")], True


def verify_chunk_encoding(tokenizer: Tokenizer, cfg: PreprocessingConfig) -> None:
    """Compare word-aligned chunk encoding with a full canonical-text encode."""
    text = canonicalize("Section 12(1) protects ₹500. Café न्यायालय. " * (cfg.encode_chunk_words + 1))
    chunked = list(itertools.chain.from_iterable(ids for ids, _ in document_token_chunks(text, tokenizer, cfg)))
    expected = [tokenizer.token_to_id("<s>")] + tokenizer.encode(text, add_special_tokens=False).ids + [tokenizer.token_to_id("</s>")]
    assert chunked == expected

[Markdown Cell]

## 16. Token-accurate source interleaving

For total training budget \(T\), require exactly \(T_s=T p_s/100\) tokens from
each source. Choose the next source by its lowest consumed fraction, then
consume a document. This preserves document boundaries and does not assume
equal document lengths. Insufficient retained capacity fails rather than
repeating a source or silently changing the mixture.

In [23]:
# [Code Cell]
def build_training(connection: sqlite3.Connection, paths: Mapping[str, Path],
                   tokenizer: Tokenizer, cfg: PreprocessingConfig) -> dict[str, Any]:
    """Write a document-interleaved memmap with exact source token quotas."""
    quotas = {source: cfg.train_tokens * TRAIN_PERCENT[source] // 100 for source in SOURCES}
    written = dict.fromkeys(SOURCES, 0)
    documents = dict.fromkeys(SOURCES, 0)
    cropped = dict.fromkeys(SOURCES, 0)
    streams = {source: iter_selected_documents(connection, paths[source], source, "split='train'", cfg)
               for source in SOURCES}
    manifest_path = cfg.output_dir / "train_documents.jsonl"
    temporary = manifest_path.with_name(manifest_path.name + ".partial")
    connection.execute("DELETE FROM training_documents")
    connection.commit()
    try:
        with AtomicTokenWriter(cfg.output_dir / "train.bin", cfg.train_tokens, cfg.vocab_size) as writer, \
                temporary.open("w", encoding="utf-8") as manifest, \
                tqdm(total=cfg.train_tokens, desc="Training mixture", unit="tok") as progress:
            while writer.written < cfg.train_tokens:
                active = [source for source in SOURCES if written[source] < quotas[source]]
                source = min(active, key=lambda item: written[item] / quotas[item])
                row, text = next(streams[source], (None, None))
                if row is None:
                    raise ValueError(f"{source} exhausted at {written[source]:,}/{quotas[source]:,} tokens. "
                                     "The retained subset cannot satisfy this quota. Use a fresh run with a smaller training budget "
                                     "or inspect the source/cache limits; source weights will not be changed.")
                assert row["split"] == "train"
                start = writer.written
                complete = False
                for ids, is_document_end in document_token_chunks(text, tokenizer, cfg, emit_eos=bool(row["fully_cached"])):
                    take = min(len(ids), quotas[source] - written[source])
                    writer.write(ids[:take])
                    written[source] += take
                    progress.update(take)
                    complete = is_document_end and take == len(ids)
                    if written[source] == quotas[source]:
                        break
                documents[source] += 1
                cropped[source] += int(not complete)
                connection.execute("INSERT INTO training_documents VALUES (?)", (row["sha256"],))
                manifest.write(json.dumps({
                    "sha256": row["sha256"], "source": source, "ordinal": row["ordinal"],
                    "cache_offset": start, "cache_length": writer.written - start,
                    "complete_document": complete, "retained_document_complete": bool(row["fully_cached"]),
                }) + "\n")
                progress.set_postfix(source=source, documents=sum(documents.values()))
                if sum(documents.values()) % 1_000 == 0:
                    connection.commit()
    finally:
        for stream in streams.values():
            stream.close()
    connection.commit()
    sync_file(temporary)
    os.replace(temporary, manifest_path)
    assert written == quotas and sum(written.values()) == cfg.train_tokens
    return {"source_tokens": written, "source_documents": documents,
            "source_cropped_documents": cropped, "total_tokens": cfg.train_tokens}

[Markdown Cell]

## 17. Full artifact verification

Check exact shapes, ID bounds, source quotas, document separation, tokenizer isolation, 512/128 word counts, every validation cache slice, and evaluator-map IDs. Scan binaries in bounded chunks.

In [24]:
# [Code Cell]
def verify_binary(path: Path, expected_count: int, vocab_size: int) -> None:
    """Validate shape, byte size, and every ID using bounded memmap slices."""
    assert path.stat().st_size == expected_count * TOKEN_DTYPE.itemsize
    cache = np.memmap(path, dtype=TOKEN_DTYPE, mode="r", shape=(expected_count,))
    try:
        assert cache.shape == (expected_count,) and cache.dtype == TOKEN_DTYPE
        for start in tqdm(range(0, expected_count, 1_000_000), desc=f"Verify {path.name}"):
            chunk = cache[start:start + 1_000_000]
            assert int(chunk.min()) >= 0 and int(chunk.max()) < vocab_size
            del chunk
    finally:
        del cache
        gc.collect()


def verify_document_manifests(connection: sqlite3.Connection, cfg: PreprocessingConfig,
                              train_stats: dict[str, Any]) -> None:
    """Check disjoint document IDs, tokenizer isolation, and training offsets."""
    assert connection.execute("SELECT COUNT(*) FROM training_documents t JOIN "
                              "validation_documents v USING (sha256)").fetchone()[0] == 0
    assert connection.execute("SELECT COUNT(*) FROM validation_documents v JOIN documents d "
                              "USING (sha256) WHERE d.sample_words>0 OR d.split!='val'").fetchone()[0] == 0
    assert connection.execute("SELECT COUNT(*) FROM training_documents t JOIN documents d "
                              "USING (sha256) WHERE d.split!='train'").fetchone()[0] == 0
    cursor = 0
    source_tokens = Counter()
    for row in iter_jsonl(cfg.output_dir / "train_documents.jsonl"):
        assert row["cache_offset"] == cursor and row["cache_length"] > 0
        cursor += row["cache_length"]
        source_tokens[row["source"]] += row["cache_length"]
    assert cursor == cfg.train_tokens and dict(source_tokens) == train_stats["source_tokens"]
    assert all(train_stats["source_tokens"][source] * 100 == cfg.train_tokens * TRAIN_PERCENT[source]
               for source in SOURCES)

In [25]:
# [Code Cell]
def verify_validation_examples(tokenizer: Tokenizer, cfg: PreprocessingConfig,
                               val_stats: dict[str, Any]) -> None:
    """Validate intact examples, corresponding gold maps, and cached token slices."""
    cache = np.memmap(cfg.output_dir / "val.bin", dtype=TOKEN_DTYPE, mode="r")
    pairs = iter_jsonl(cfg.output_dir / "validation_pairs.jsonl")
    prompts = iter_json_records(cfg.output_dir / "validation_prompts.json")
    truth = iter_json_records(cfg.output_dir / "validation_truth.json")
    cached_by_source, examples_by_source = Counter(), Counter()
    seen_ids: set[str] = set()  # Only the bounded validation example IDs.
    cursor = 0
    try:
        for pair, prompt_row, truth_row in itertools.zip_longest(pairs, prompts, truth):
            assert pair is not None and prompt_row is not None and truth_row is not None
            assert pair["id"] not in seen_ids
            seen_ids.add(pair["id"])
            assert len(pair["prompt"].split()) == cfg.prompt_words
            assert len(pair["reference"].split()) == cfg.reference_words
            assert prompt_row[1] == truth_row[1] == pair["id"]
            assert prompt_row[2]["text"] == pair["prompt"]
            assert truth_row[2]["text"] == pair["reference"]
            ids = [tokenizer.token_to_id("<s>")] + tokenizer.encode(
                pair["prompt"] + " " + pair["reference"], add_special_tokens=False
            ).ids + [tokenizer.token_to_id("</s>")]
            length = pair["cache_length"]
            assert 0 < length <= len(ids) == pair["full_token_length"]
            assert pair["cache_offset"] == cursor
            assert pair["cache_truncated"] == (length < len(ids))
            assert np.array_equal(cache[cursor:cursor + length], np.asarray(ids[:length], dtype=TOKEN_DTYPE))
            cursor += length
            cached_by_source[pair["source"]] += length
            examples_by_source[pair["source"]] += 1
        assert cursor == cfg.val_tokens
        for source in SOURCES:
            assert cached_by_source[source] == cfg.val_tokens // len(SOURCES)
            assert examples_by_source[source] == val_stats[source]["examples"]
    finally:
        pairs.close()
        prompts.close()
        truth.close()
        del cache
        gc.collect()

[Markdown Cell]

## 18. Durable state and a single active run

Bind state to configuration, package versions, and the immutable source
manifest. A lock prevents two runs in the same output directory. A dead
kernel's lock can be reclaimed after verifying its PID is no longer alive;
the lock holder is never automatically interrupted. Keep `_preprocess/` for
resume. Kaggle session files survive only if you preserve/download/save those
outputs yourself; code cannot recover files the platform has discarded.

Changing budgets/configuration requires a fresh output subdirectory or an
explicit cleanup of this job's generated files. A token rotation does not
change the data identity. Completed binary stages are checked by file hashes.

In [26]:
# [Code Cell]
def process_identity(pid: int) -> dict[str, str] | None:
    """Identify Linux processes without signalling any Windows process."""
    if os.name == "posix" and Path("/proc").is_dir():
        try:
            boot = Path("/proc/sys/kernel/random/boot_id").read_text().strip()
            fields = Path(f"/proc/{pid}/stat").read_text().rsplit(") ", 1)[1].split()
            return {"boot": boot, "start_ticks": fields[19]}
        except FileNotFoundError:
            return None
    if pid == os.getpid():
        return {"pid": str(pid)}
    raise RuntimeError("Cannot verify a stale PID on this platform; confirm no run is active before removing its lock.")


@contextmanager
def run_lock(cfg: PreprocessingConfig) -> Iterator[None]:
    """Prevent concurrent writers and recover only a demonstrably dead PID."""
    path = cfg.output_dir / "_preprocess" / "run.lock"
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        try:
            owner = json.loads(path.read_text(encoding="utf-8"))
            current = process_identity(int(owner["pid"]))
        except (ValueError, KeyError, json.JSONDecodeError, PermissionError):
            raise RuntimeError("Cannot verify the existing run lock. Remove it only after confirming no run is active.") from None
        else:
            if current is not None and current == owner.get("identity"):
                raise RuntimeError("Another live process owns this output directory. Stop that run or use a different directory.")
            if current is not None and "identity" not in owner:
                raise RuntimeError("Old lock identity is ambiguous; verify no run is active before removing it.")
            path.unlink()
    try:
        with path.open("x", encoding="utf-8") as stream:
            json.dump({"pid": os.getpid(), "identity": process_identity(os.getpid()), "created": time.time()}, stream)
        yield
    finally:
        if path.exists():
            try:
                if json.loads(path.read_text(encoding="utf-8"))["pid"] == os.getpid():
                    path.unlink()
            except (ValueError, KeyError, json.JSONDecodeError):
                pass


def initialize_run(cfg: PreprocessingConfig) -> tuple[Path, dict[str, Any], dict[str, SourceSpec]]:
    """Reuse the saved immutable manifest without another remote metadata call."""
    work = cfg.output_dir / "_preprocess"
    work.mkdir(parents=True, exist_ok=True)
    identity = {"pipeline_version": PREPROCESS_VERSION, "config": asdict(cfg),
                "tokenizers": importlib.metadata.version("tokenizers"),
                "ijson": importlib.metadata.version("ijson")}
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True, default=str).encode()).hexdigest()
    state_path = work / "state.json"
    if state_path.exists():
        state = json.loads(state_path.read_text(encoding="utf-8"))
        if state["fingerprint"] != fingerprint:
            raise RuntimeError("Saved state has different inputs/configuration or pipeline version. "
                               "Use a fresh output directory; do not reuse stale artifacts.")
        sources = {key: SourceSpec(**value) for key, value in state["sources"].items()}
    else:
        sources = discover_sources(cfg)
        state = {"fingerprint": fingerprint, "identity": identity,
                 "sources": {key: asdict(value) for key, value in sources.items()}, "stages": {}}
        atomic_json(state_path, state)
    return state_path, state, sources

In [27]:
# [Code Cell]
def completed_stage(state: dict[str, Any], stage: str, paths: Sequence[Path]) -> bool:
    """Reuse only stages whose committed file hashes still match."""
    saved = state["stages"].get(stage)
    if saved is None:
        return False
    return all(path.is_file() and file_sha256(path) == saved["files"].get(path.name) for path in paths)


def commit_stage(state_path: Path, state: dict[str, Any], stage: str,
                 result: Any, paths: Sequence[Path]) -> None:
    """Publish a stage receipt after all its own outputs are durable."""
    state["stages"][stage] = {"result": result,
                              "files": {path.name: file_sha256(path) for path in paths}}
    atomic_json(state_path, state)

[Markdown Cell]

## 19. Complete or resume ingestion

Only incomplete sources access the network. Compressed data is verified when reused. SQLite failure rolls back the current batch, and resume starts from the last committed record count.

In [28]:
# [Code Cell]
def prepare_index(cfg: PreprocessingConfig, sources: Mapping[str, SourceSpec],
                  state_path: Path, state: dict[str, Any]) -> tuple[sqlite3.Connection, dict[str, Any]]:
    """Resume ingestion, publish its audit receipt, then plan local BPE sampling."""
    database_path = state_path.parent / "documents.sqlite"
    if not database_path.exists():
        state["stages"] = {}
        atomic_json(state_path, state)
    connection = open_database(database_path, cfg)
    try:
        create_index(connection)
        if connection.execute("PRAGMA quick_check").fetchone()[0] != "ok":
            raise RuntimeError("SQLite integrity check failed; restore a saved index or start a fresh run.")
        pending = connection.execute("SELECT COUNT(*) FROM source_progress WHERE complete=1").fetchone()[0] != 4
        if pending:
            state["stages"] = {}
            atomic_json(state_path, state)
        audit = {source: ingest_source(connection, sources[source], cfg) for source in SOURCES}
        audit_path = cfg.output_dir / "audit.json"
        atomic_json(audit_path, audit)
        commit_stage(state_path, state, "audit", audit, [audit_path])
        if "groups" not in state["stages"]:
            plan = plan_document_groups(connection, cfg)
            commit_stage(state_path, state, "groups", plan, [])
        return connection, audit
    except sqlite3.OperationalError as exc:
        connection.rollback()
        connection.close()
        if "full" in str(exc).lower():
            raise OSError("SQLite reached its physical storage cap. Lower train_tokens for a fresh run "
                          "or increase caps only if actual Kaggle disk space permits.") from None
        raise
    except BaseException:
        connection.close()
        raise

In [29]:
# [Code Cell]
def prepare_tokenizer(connection: sqlite3.Connection, cfg: PreprocessingConfig,
                      paths: Mapping[str, Path], state_path: Path,
                      state: dict[str, Any]) -> tuple[Tokenizer, dict[str, int]]:
    """Load or train BPE, invalidating caches if its artifact is regenerated."""
    tokenizer_path = cfg.output_dir / "tokenizer.json"
    if completed_stage(state, "tokenizer", [tokenizer_path]):
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
        print("Reusing completed BPE tokenizer.")
    else:
        for stage in ("tokenizer", "validation", "training"):
            state["stages"].pop(stage, None)
        atomic_json(state_path, state)
        tokenizer = train_legal_tokenizer(connection, paths, tokenizer_path, cfg)
        commit_stage(state_path, state, "tokenizer", verify_tokenizer(tokenizer, cfg), [tokenizer_path])
    special_ids = verify_tokenizer(tokenizer, cfg)
    verify_chunk_encoding(tokenizer, cfg)
    return tokenizer, special_ids

In [30]:
# [Code Cell]
def prepare_caches(connection: sqlite3.Connection, cfg: PreprocessingConfig,
                   paths: Mapping[str, Path], tokenizer: Tokenizer,
                   state_path: Path, state: dict[str, Any]) -> tuple[dict[str, Any], dict[str, Any]]:
    """Build or reuse verified stage files in dependency order."""
    validation_paths = [cfg.output_dir / name for name in (
        "val.bin", "validation_pairs.jsonl", "validation_documents.jsonl",
        "validation_prompts.json", "validation_truth.json",
    )]
    if completed_stage(state, "validation", validation_paths):
        val_stats = state["stages"]["validation"]["result"]
        print("Reusing completed validation cache and generation examples.")
    else:
        state["stages"].pop("validation", None)
        state["stages"].pop("training", None)
        atomic_json(state_path, state)
        ensure_disk_space(cfg.output_dir, cfg.val_tokens * TOKEN_DTYPE.itemsize, cfg)
        val_stats = build_validation(connection, paths, tokenizer, cfg)
        export_evaluation_map(validation_paths[1], validation_paths[3], "prompt")
        export_evaluation_map(validation_paths[1], validation_paths[4], "reference")
        commit_stage(state_path, state, "validation", val_stats, validation_paths)
    training_paths = [cfg.output_dir / name for name in ("train.bin", "train_documents.jsonl")]
    if completed_stage(state, "training", training_paths):
        train_stats = state["stages"]["training"]["result"]
        print("Reusing completed training cache.")
    else:
        state["stages"].pop("training", None)
        atomic_json(state_path, state)
        ensure_disk_space(cfg.output_dir, cfg.train_tokens * TOKEN_DTYPE.itemsize, cfg)
        train_stats = build_training(connection, paths, tokenizer, cfg)
        commit_stage(state_path, state, "training", train_stats, training_paths)
    return train_stats, val_stats

[Markdown Cell]

## 20. Publish a verified downstream contract

Export raw token caches and complete continuation examples at the dataset
root, with vocabulary IDs, exact quotas, audit scope, source commit/size
identities, storage usage, and artifact checksums. The tokenizer and caches
are self-contained once published; training never needs the 21GB raw inputs.

In [31]:
# [Code Cell]
def publish_metadata(connection: sqlite3.Connection, tokenizer: Tokenizer,
                     special_ids: dict[str, int], cfg: PreprocessingConfig,
                     state: dict[str, Any], audit: dict[str, Any],
                     train_stats: dict[str, Any], val_stats: dict[str, Any]) -> dict[str, Any]:
    """Verify every required contract and atomically publish the final receipt."""
    verify_binary(cfg.output_dir / "train.bin", cfg.train_tokens, cfg.vocab_size)
    verify_binary(cfg.output_dir / "val.bin", cfg.val_tokens, cfg.vocab_size)
    verify_document_manifests(connection, cfg, train_stats)
    verify_validation_examples(tokenizer, cfg, val_stats)
    metadata = {
        "status": "verified", "schema_version": 1, "pipeline_version": PREPROCESS_VERSION,
        "run_fingerprint": state["fingerprint"], "config": asdict(cfg),
        "inputs": state["sources"], "dtype": TOKEN_DTYPE.str,
        "audit_scope": "full corpus" if cfg.audit_full_corpus else "budgeted streamed prefixes",
        "storage_bytes_before_metadata": job_disk_bytes(cfg),
        "vocab_size": cfg.vocab_size, "special_token_ids": special_ids,
        "train_tokens": cfg.train_tokens, "val_tokens": cfg.val_tokens,
        "training": train_stats, "validation": val_stats, "audit": audit,
        "tokenizer_sampling": state["stages"]["groups"]["result"],
        "policies": {
            "normalization": "Unicode NFC; collapse whitespace; preserve case and punctuation",
            "deduplication": "global SHA-256 of canonical UTF-8 document text",
            "duplicate_source_ownership_order": list(SOURCES),
            "split_unit": "whole original unique document; unused retained candidates return to training",
            "validation_source_percent": dict.fromkeys(SOURCES, 25),
            "validation_windows": "nonoverlapping 512-word prompt + 128-word reference",
            "cache_special_tokens": "one BOS/EOS per document or validation window; no padding",
            "budget_cropping": "last source document/window can be cropped; JSON examples stay intact",
            "training_source_percent": TRAIN_PERCENT,
            "training_sampling": "without replacement; document-level source interleaving",
            "tokenizer_data": "seeded bounded excerpts from retained train text; candidates excluded",
            "raw_input_storage": "none in HF mode; bounded compressed selected text only",
            "resume": "transactional record checkpoints for ingestion; committed prefix replay after kernel restart; atomic output stages",
        },
        "runtime": {"python": platform.python_version(), "numpy": np.__version__,
                    "tokenizers": importlib.metadata.version("tokenizers"),
                    "ijson": importlib.metadata.version("ijson"),
                    "huggingface_hub": importlib.metadata.version("huggingface-hub"),
                    "requests": importlib.metadata.version("requests")},
        "downstream_input_root": "/kaggle/input/cs781-preprocessed-data/",
        "artifacts": {name: {"bytes": (cfg.output_dir / name).stat().st_size,
                             "sha256": file_sha256(cfg.output_dir / name)} for name in FINAL_ARTIFACTS},
    }
    atomic_json(cfg.output_dir / "metadata.json", metadata)
    return metadata

In [32]:
# [Code Cell]
def run_preprocessing(cfg: PreprocessingConfig) -> dict[str, Any]:
    """Run bounded streaming ingestion and verified local-cache preprocessing."""
    cfg.validate()
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    with run_lock(cfg):
        ensure_disk_space(cfg.output_dir, 0, cfg)
        state_path, state, sources = initialize_run(cfg)
        metadata_path = cfg.output_dir / "metadata.json"
        if metadata_path.exists():
            metadata_path.unlink()
        connection, audit = prepare_index(cfg, sources, state_path, state)
        try:
            tokenizer, special_ids = prepare_tokenizer(connection, cfg, sources, state_path, state)
            train_stats, val_stats = prepare_caches(connection, cfg, sources, tokenizer, state_path, state)
            metadata = publish_metadata(connection, tokenizer, special_ids, cfg, state, audit, train_stats, val_stats)
            ensure_disk_space(cfg.output_dir, 0, cfg)
        finally:
            connection.close()
    return metadata


metadata = run_preprocessing(cfg)
print("Verified artifacts:", cfg.output_dir)
print("Audit scope:", metadata["audit_scope"])
print("Training source tokens:", metadata["training"]["source_tokens"])
print("Validation source tokens:", {source: metadata["validation"][source]["cached_tokens"] for source in SOURCES})
print("Job disk use:", round(job_disk_bytes(cfg) / 2**30, 3), "GiB")

Pinned HF revision: 89edba6a44d6bb9593fd13b6895b5cd3b06eb82e
Raw source total (not stored locally): 17.89 GiB


Stream supreme_court: 0doc [00:00, ?doc/s]

Stream high_courts: 0doc [00:00, ?doc/s]

Stream acts: 0doc [00:00, ?doc/s]

Stream tribunals: 0doc [00:00, ?doc/s]

BPE sample:   0%|          | 0/4000000 [00:00<?, ?word/s]

... (4 outputs left out)

Training mixture:   0%|          | 0/200000000 [00:00<?, ?tok/s]

Verify train.bin:   0%|          | 0/200 [00:00<?, ?it/s]

Verify val.bin:   0%|          | 0/5 [00:00<?, ?it/s]

Verified artifacts: /kaggle/working
Audit scope: budgeted streamed prefixes
Training source tokens: {'supreme_court': 70000000, 'high_courts': 70000000, 'acts': 30000000, 'tribunals': 30000000}
Validation source tokens: {'supreme_court': 1250000, 'high_courts': 1250000, 'acts': 1250000, 'tribunals': 1250000}
Job disk use: 1.085 GiB


[Markdown Cell]

## 21. Reopen final exports

Check the actual files that independent training notebooks will consume.

In [33]:
# [Code Cell]
saved_metadata = json.loads((cfg.output_dir / "metadata.json").read_text(encoding="utf-8"))
saved_tokenizer = Tokenizer.from_file(str(cfg.output_dir / "tokenizer.json"))
assert saved_metadata["status"] == "verified"
assert saved_metadata["dtype"] == "<u2"
assert saved_tokenizer.get_vocab_size() == 32_768
assert saved_metadata["train_tokens"] == cfg.train_tokens
assert saved_metadata["val_tokens"] == cfg.val_tokens
assert saved_metadata["special_token_ids"] == verify_tokenizer(saved_tokenizer, cfg)
for name, receipt in saved_metadata["artifacts"].items():
    path = cfg.output_dir / name
    assert path.stat().st_size == receipt["bytes"]
    assert file_sha256(path) == receipt["sha256"]
for name, token_count in (("train.bin", cfg.train_tokens), ("val.bin", cfg.val_tokens)):
    cache = np.memmap(cfg.output_dir / name, dtype=TOKEN_DTYPE, mode="r")
    assert cache.shape == (token_count,)
    print(name, "shape=", cache.shape, "dtype=", cache.dtype, "first IDs=", cache[:12].tolist())
    del cache
gc.collect()
print("Exported artifacts reopened and verified successfully.")

train.bin shape= (200000000,) dtype= uint16 first IDs= [1, 5583, 31, 8079, 372, 15931, 1078, 9151, 22149, 553, 15786, 60]
val.bin shape= (5000000,) dtype= uint16 first IDs= [1, 5583, 31, 31938, 3106, 18182, 303, 17875, 8944, 13551, 1078, 1082]
Exported artifacts reopened and verified successfully.


[Markdown Cell]

## 22. Handoff and operational cases

Save the ten final root artifacts, including `metadata.json`, as Kaggle Dataset
`cs781-preprocessed-data`. Training reads
`/kaggle/input/cs781-preprocessed-data/` and writes checkpoints to
`/kaggle/working/`. Keep `_preprocess/` in a separate preserved output if you
want to resume preprocessing; training notebooks do not need it.

| Situation | Behavior |
|---|---|
| Missing token / gating not accepted / 401–403 | Clear setup error; no token is printed |
| Missing or ambiguous repository filenames | Require exact `hf_files` overrides |
| `main` changes during/after a run | Reuse the saved immutable commit |
| HTTP 429, 5xx, timeout, dropped connection | Bounded retry; resume delivered-byte offset |
| Server ignores Range | Bounded prefix discard; parser receives no duplicate bytes |
| Wrong Content-Range, HTML access page, unexpected compression | Reject the response |
| Kernel interruption during ingestion | Replay unfinished source; verify committed document chain |
| Completed ingestion, Internet disabled | Build/reuse local stages without reading secrets |
| Disk/cache/database cap or huge document | Stop safely with an actionable limit error |
| Source cannot fill its validation/train token quota | Fail; never alter proportions or repeat a source |
| Corrupted published cache | Hash mismatch invalidates dependent output stages |
| Corrupted retained text or SQLite | Integrity check fails; restore state or use a fresh run |
| Malformed JSON beyond a budgeted prefix | Not inspected; choose full auditing to inspect every record |
| Kaggle has discarded previous working files | Restore saved outputs; automatic recovery is impossible |

Whole-document exact deduplication does not claim semantic/near-duplicate
removal. The BPE trainer's vocabulary-size assertion, all export checks, and
the held-out document assertions must pass before any model training.

Stop after Notebook 00 and confirm before Notebook 01.

In [36]:
!kaggle kernels output shubhamjoshi1806/data-and-tokenizer -p /kaggle/working/downloaded_output

Output file downloaded to /kaggle/working/downloaded_output/.virtual_documents/__notebook_source__.ipynb
Output file downloaded to /kaggle/working/downloaded_output/_preprocess/documents.sqlite
Output file downloaded to /kaggle/working/downloaded_output/_preprocess/state.json
Output file downloaded to /kaggle/working/downloaded_output/audit.json
Output file downloaded to /kaggle/working/downloaded_output/metadata.json
Output file downloaded to /kaggle/working/downloaded_output/tokenizer.json
Output file downloaded to /kaggle/working/downloaded_output/train.bin
Output file downloaded to /kaggle/working/downloaded_output/train_documents.jsonl
Output file downloaded to /kaggle/working/downloaded_output/val.bin
Output file downloaded to /kaggle/working/downloaded_output/validation_documents.jsonl
Output file downloaded to /kaggle/working/downloaded_output/validation_pairs.jsonl
Output file downloaded to /kaggle/working/downloaded_output/validation_prompts.json
Output file downloaded to /ka

In [38]:
!zip -r /kaggle/working/data-and-tokenizer-output.zip /kaggle/working/downloaded_output

updating: kaggle/working/downloaded_output/ (stored 0%)
updating: kaggle/working/downloaded_output/audit.json (deflated 69%)
updating: kaggle/working/downloaded_output/validation_prompts.json (deflated 72%)
updating: kaggle/working/downloaded_output/tokenizer.json (deflated 82%)
updating: kaggle/working/downloaded_output/.virtual_documents/ (stored 0%)
updating: kaggle/working/downloaded_output/.virtual_documents/__notebook_source__.ipynb (deflated 73%)
updating: kaggle/working/downloaded_output/_preprocess/ (stored 0%)
updating: kaggle/working/downloaded_output/_preprocess/documents.sqlite (deflated 18%)
updating: kaggle/working/downloaded_output/_preprocess/state.json (deflated 72%)
updating: kaggle/working/downloaded_output/validation_pairs.jsonl (deflated 74%)
updating: kaggle/working/downloaded_output/validation_documents.jsonl (deflated 69%)
updating: kaggle/working/downloaded_output/metadata.json (deflated 68%)
updating: kaggle/working/downloaded_output/train.bin (deflated 49%)
